.stream() (synchronous) and .astream() (asynchronous), along with the two stream_mode options ("updates" vs. "values") and astream_events.

1. Simple Explanation: stream vs. astream & stream_mode
1.1invoke() vs. stream():

   graph.invoke(...) waits until every single node in the graph finishes, then dumps the entire final result at once.

   graph.stream(...) yields intermediate outputs live, step-by-step, right as each node finishes running (for example, showing the AI's tool call first, then the tool output, then the final answer).

1.2stream() vs. astream():

   graph.stream(...) is synchronous (normal for loop)—it blocks Python while waiting for each step.

   graph.astream(...) is asynchronous (async for loop)—used in async apps (like FastAPI or Jupyter async cells) so Python doesn't freeze while waiting on API calls, and it also unlocks astream_events to stream token-by-token like ChatGPT typing live on screen.

1.3stream_mode="updates" vs. stream_mode="values":

   "updates" (default): Shows only what changed in State after each node ran (just the new message added by that node).

   "values": Shows the entire State dictionary (every message from the very beginning up to that step) after each node finishes.

In [2]:
from typing import Annotated
from typing_extensions import TypedDict
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch
from langchain_core.tools import tool

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

load_dotenv(override=True)

# 1. Tools
tavily_tool = TavilySearch(max_results=2)

@tool
def multiply(a: int, b: int) -> int:
    """Multiply a and b."""
    return a * b

tools = [tavily_tool, multiply]

# 2. Gemini Model with Tools
llm = init_chat_model("google_genai:gemini-3.5-flash-lite")
llm_with_tools = llm.bind_tools(tools)

# 3. State & Node
class State(TypedDict):
    messages: Annotated[list, add_messages]

def tool_calling_llm(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

# 4. Build & Compile Graph with MemorySaver
builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges("tool_calling_llm", tools_condition)
builder.add_edge("tools", "tool_calling_llm")

memory = MemorySaver()
graph = builder.compile(checkpointer=memory)

In [3]:
config = {"configurable": {"thread_id": "3"}}

for chunk in graph.stream(
    {"messages": "Hi, my name is Harshitha and I like coding and playing badminton"},
    config,
    stream_mode="updates",
):
    print(chunk)

{'tool_calling_llm': {'messages': [AIMessage(content=[{'type': 'text', 'text': 'Nice to meet you, Harshitha! Coding and playing badminton are a great combination—one gives your brain a nice analytical workout, and the other keeps you physically active. \n\nWhat kind of coding do you usually like to do (like web development, apps, or algorithms)? And how often do you get to hit the badminton court?', 'extras': {'signature': 'EpYBCpMBAWkUfRMhD/Gvq2XW4Sn3DJRNPqRq9BQ3UQl7Vy05LsbII4wom8DIJHCGRIFDMEI9tzDEYfxjGtCu8uhzM8lETDhAO4o59QPWLX+SDjsGBb0L/8ZislzDeufMebFWsxrOIrO1muhirsLff6OQsKENUop1B1OKvmXpLi6bWkAEYDSoL5rXmsfn6Ex6ZW8oPhKltZc3'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'normalized_finish_reason': 'stop', 'model_provider': 'google_genai'}, id='lc_run--01a20ede-8cc5-7764-9416-e6f30e9c0f7f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1557, 'output_tokens': 68, 'total_tokens'

In [4]:
# Create a fresh thread_id for streaming
config = {"configurable": {"thread_id": "3"}}

for chunk in graph.stream(
    {"messages": "Hi, my name is Harshitha and I like coding and playing badminton"},
    config,
    stream_mode="updates",
):
    print(chunk)

{'tool_calling_llm': {'messages': [AIMessage(content=[{'type': 'text', 'text': "Hi again, Harshitha! It looks like you might have sent that message twice, but I'm happy to hear it again. \n\nSince you're into both coding and badminton, have you ever thought about combining them? For instance, you could build a shuttlecock tracker, a badminton tournament management app, or even a simple game! \n\nWhat projects or technologies are you currently working on with your coding?", 'extras': {'signature': 'EpYBCpMBAWkUfRP3dJEJL5RFuF5uQX99/X46dGL1qbRgriag/mM3QO5yx0gvo7Xe/V93n37220BpMs1sogph1X92UaLKvEfp1yASPWfQikuzf7bIA7Yu+UPQd2lm3F9dxaz5PDzLbgTx2vbxwQ7sDVmTUeTdf2GCvuAg12yYoIwBJ/95gCcbzswXzq9lnhbNjL/CzEUWxjlC'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'normalized_finish_reason': 'stop', 'model_provider': 'google_genai'}, id='lc_run--01a20ede-e990-7078-9d0a-f4ecce71dcfd-0', tool_calls=[], invalid_tool_calls=[]

In [5]:
for chunk in graph.stream(
    {"messages": "I also like watching cricket"},
    config,
    stream_mode="values",
):
    print(chunk)

{'messages': [HumanMessage(content='Hi, my name is Harshitha and I like coding and playing badminton', additional_kwargs={}, response_metadata={}, id='4672f3a6-9cad-4516-8f29-1074033a63cf'), AIMessage(content=[{'type': 'text', 'text': 'Nice to meet you, Harshitha! Coding and playing badminton are a great combination—one gives your brain a nice analytical workout, and the other keeps you physically active. \n\nWhat kind of coding do you usually like to do (like web development, apps, or algorithms)? And how often do you get to hit the badminton court?', 'extras': {'signature': 'EpYBCpMBAWkUfRMhD/Gvq2XW4Sn3DJRNPqRq9BQ3UQl7Vy05LsbII4wom8DIJHCGRIFDMEI9tzDEYfxjGtCu8uhzM8lETDhAO4o59QPWLX+SDjsGBb0L/8ZislzDeufMebFWsxrOIrO1muhirsLff6OQsKENUop1B1OKvmXpLi6bWkAEYDSoL5rXmsfn6Ex6ZW8oPhKltZc3'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'normalized_finish_reason': 'stop', 'model_provider': 'google_genai'}, id='lc_r

In [6]:
config = {"configurable": {"thread_id": "4"}}

async for chunk in graph.astream(
    {"messages": "Hi, my name is Harshitha and I like coding"},
    config,
    stream_mode="updates",
):
    print(chunk)

{'tool_calling_llm': {'messages': [AIMessage(content=[{'type': 'text', 'text': "Hi Harshitha! It's nice to meet you. Coding is a lot of fun—what kind of programming languages or projects do you usually work on?", 'extras': {'signature': 'EpYBCpMBAWkUfROGrX3YJkteqil8TeenuJ0QIdnnuo9aSUhyWUrKzsYZI1qogt1lijr6i6zBpWxXTl1u7AHJi05imR3xcvHzdg2Bw5sAhgnUtHZwLt2F682DFLCfK4TNG3Mt2AVxBNsn3/SznncMN13VJjQmDP0PJ/KOOJV6jE7g/NSMb+C6/s2p6ggkZNUk8Z6R8C3omYKd'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'normalized_finish_reason': 'stop', 'model_provider': 'google_genai'}, id='lc_run--01a20edf-86da-7497-b615-44fb6b40e340-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1554, 'output_tokens': 32, 'total_tokens': 1586, 'input_token_details': {'cache_read': 0}})]}}


In [7]:
config = {"configurable": {"thread_id": "5"}}

async for event in graph.astream_events(
    {"messages": "Hi, my name is Harshitha and I like coding"},
    config,
    version="v2",
):
    print(event)

{'event': 'on_chain_start', 'data': {'input': {'messages': 'Hi, my name is Harshitha and I like coding'}}, 'name': 'LangGraph', 'tags': [], 'run_id': '01a20edf-dbdc-76b8-93c3-7a6f03f63773', 'metadata': {'thread_id': '5', 'ls_integration': 'langgraph'}, 'parent_ids': []}
{'event': 'on_chain_start', 'data': {'input': {'messages': [HumanMessage(content='Hi, my name is Harshitha and I like coding', additional_kwargs={}, response_metadata={}, id='91a65188-677c-40cd-96cc-fe562cc10656')]}}, 'name': 'tool_calling_llm', 'tags': ['graph:step:1'], 'run_id': '01a20edf-dbdf-775d-b612-ae993622ca16', 'metadata': {'thread_id': '5', 'ls_integration': 'langgraph', 'langgraph_step': 1, 'langgraph_node': 'tool_calling_llm', 'langgraph_triggers': ('branch:to:tool_calling_llm',), 'langgraph_path': ('__pregel_pull', 'tool_calling_llm'), 'langgraph_checkpoint_ns': 'tool_calling_llm:b0747da0-f287-e661-b667-25f607662311'}, 'parent_ids': ['01a20edf-dbdc-76b8-93c3-7a6f03f63773']}
{'event': 'on_chat_model_start', 